# Dashboard Data Preparation

This notebook prepares aggregated data tables and KPI values for the dashboard.
It loads the processed data from 03_analysis_and_visuals and saves clean, dashboard-ready tables to data/processed/

## 1. Import Required Libraries

In [3]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded successfully")

Libraries loaded successfully


## 2. Load Raw Data and Clean

In [4]:
print("Loading raw data...")
df = pd.read_pickle('../data/raw/AT.pkl')

# Rename columns (from data cleaning notebook)
column_rename_map = {
    'Company name Latin alphabet': 'company_name',
    'Country ISO code': 'country_code', 
    'City\nLatin Alphabet': 'city',
    'NACE Rev. 2, core code (4 digits)': 'nace_code',
    'BvD ID number': 'bvd_id',
    'NACE Rev. 2 main section': 'nace_section',
    'Region in country': 'region_raw',
    'Status': 'status',
    'Date of incorporation': 'incorporation_date'
}

df = df.rename(columns=column_rename_map)

# Add employee columns
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
for year in years:
    raw_col = f'Number of employees\n{year}'
    num_col = f'emp_{year}_num'
    if raw_col in df.columns:
        df[num_col] = pd.to_numeric(df[raw_col], errors='coerce')

# Add founded_year
founded_col = 'Founded Year'
if founded_col in df.columns:
    df['founded_year'] = pd.to_datetime(df[founded_col], errors='coerce').dt.year

# Clean region
region_col = 'Region in country clean'
if region_col in df.columns:
    df['region'] = df[region_col].str.strip()

print(f"Data loaded and cleaned: {df.shape}")
print(f"Total firms: {len(df):,}")

# Calculate growth variables
print("\nCalculating growth variables...")

def safe_growth(emp_current, emp_previous):
    """Calculate growth rate, returning 'n.a.' if either input is missing"""
    if pd.isna(emp_current) or pd.isna(emp_previous) or emp_previous == 0:
        return 'n.a.'
    return (emp_current - emp_previous) / emp_previous

# Calculate year-over-year growth
df['growth_2024'] = df.apply(lambda row: safe_growth(row['emp_2024_num'], row['emp_2023_num']), axis=1)
df['growth_2023'] = df.apply(lambda row: safe_growth(row['emp_2023_num'], row['emp_2022_num']), axis=1)  
df['growth_2022'] = df.apply(lambda row: safe_growth(row['emp_2022_num'], row['emp_2021_num']), axis=1)

def calculate_aagr(growth_2024, growth_2023, growth_2022):
    """Calculate AAGR from three growth rates"""
    growths = [growth_2024, growth_2023, growth_2022]
    if any(g == 'n.a.' for g in growths):
        return 'n.a.'
    product = 1
    for g in growths:
        product *= (1 + g)
    return product**(1/3) - 1

df['aagr_2024'] = df.apply(lambda row: calculate_aagr(row['growth_2024'], row['growth_2023'], row['growth_2022']), axis=1)

# Apply classification
print("Applying high-growth firm classification...")

growth_cols = ['growth_2024', 'growth_2023', 'growth_2022', 'aagr_2024']
for col in growth_cols:
    df[f'{col}_num'] = pd.to_numeric(df[col], errors='coerce')

def classify_high_growth_firm(row):
    """Classify as Consistent High Growth Firm following Belgium methodology"""
    if pd.isna(row['emp_2021_num']) or row['emp_2021_num'] < 10:
        return 'n.a.'
    
    required_growth = [row['growth_2024_num'], row['growth_2023_num'], row['growth_2022_num']]
    if any(pd.isna(g) for g in required_growth):
        return 'n.a.'
    
    avg_growth = sum(required_growth) / len(required_growth)
    if avg_growth > 0.10:
        return 1
    else:
        return 0

df['ConsistentHighGrowthFirm_2024'] = df.apply(classify_high_growth_firm, axis=1)

print("Data processing complete!")

Loading raw data...
Data loaded and cleaned: (46085, 30)
Total firms: 46,085

Calculating growth variables...
Applying high-growth firm classification...
Data processing complete!


## 3. Create Master Company-Level Table

In [5]:
# Create master table with classified firms only
master_table = df[df['ConsistentHighGrowthFirm_2024'] != 'n.a.'].copy()

# Select key columns for dashboard
key_columns = [
    'company_name', 'city', 'region', 'nace_section',
    'founded_year', 'emp_2021_num', 'emp_2022_num', 'emp_2023_num', 'emp_2024_num',
    'growth_2022_num', 'growth_2023_num', 'growth_2024_num', 'aagr_2024_num',
    'ConsistentHighGrowthFirm_2024'
]

available_columns = [col for col in key_columns if col in master_table.columns]
master_table = master_table[available_columns]

# Rename for clarity
master_table = master_table.rename(columns={
    'ConsistentHighGrowthFirm_2024': 'high_growth_classification',
    'emp_2021_num': 'employees_2021',
    'emp_2022_num': 'employees_2022',
    'emp_2023_num': 'employees_2023',
    'emp_2024_num': 'employees_2024',
    'growth_2022_num': 'growth_2022',
    'growth_2023_num': 'growth_2023',
    'growth_2024_num': 'growth_2024',
    'aagr_2024_num': 'aagr_2024'
})

print(f"Master table created: {master_table.shape}")
print(f"High-growth firms: {(master_table['high_growth_classification'] == 1).sum():,}")
print(f"Not high-growth firms: {(master_table['high_growth_classification'] == 0).sum():,}")
print("\nFirst few rows:")
print(master_table.head())

Master table created: (18629, 14)
High-growth firms: 1,883
Not high-growth firms: 16,746

First few rows:
                                        company_name     city          region  \
1                             OMV AKTIENGESELLSCHAFT     WIEN            Wien   
2                   OMV GAS MARKETING & TRADING GMBH     WIEN            Wien   
3                                         STRABAG SE  VILLACH         Karnten   
4                                     VOESTALPINE AG     LINZ  Oberosterreich   
7  OESTERREICHISCHE BUNDESBAHNEN-HOLDING AKTIENGE...     WIEN            Wien   

                                        nace_section  founded_year  \
1                                  C - Manufacturing        1970.0   
2  D - Electricity, gas, steam and air conditioni...        1970.0   
3                                   F - Construction        1970.0   
4                                  C - Manufacturing        1970.0   
7             K - Financial and insurance activities     

## 4. Create Industry Summary Table

In [6]:
# Group by industry (NACE section)
industry_summary = df.groupby('nace_section').agg(
    total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
    classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
    high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum()),
    mean_aagr=('aagr_2024', lambda x: x[x != 'n.a.'].astype(float).mean())
).reset_index()

# Calculate share of high-growth firms (among classified)
industry_summary['high_growth_share'] = industry_summary['high_growth_firms'] / industry_summary['classified_firms']

# Sort by number of firms descending
industry_summary = industry_summary.sort_values('total_firms', ascending=False)

industry_summary.columns = ['industry', 'total_firms', 'classified_firms', 'high_growth_firms', 'mean_aagr', 'high_growth_share']

print("Industry Summary (Top 10):")
print(industry_summary.head(10).to_string())
print(f"\nTotal industries: {len(industry_summary)}")

Industry Summary (Top 10):
                                                                    industry  total_firms  classified_firms  high_growth_firms  mean_aagr  high_growth_share
6   G - Wholesale and retail trade; repair of motor vehicles and motorcycles         9700              4510                366   0.032415           0.081153
5                                                           F - Construction         7564              3662                340   0.041094           0.092845
2                                                          C - Manufacturing         6862              3768                330   0.035755           0.087580
8                              I - Accommodation and food service activities         4989              1171                142   0.070302           0.121264
12                     M - Professional, scientific and technical activities         4035               905                145   0.079850           0.160221
13                         N - 

## 5. Create Region Summary Table

In [7]:
# Group by region
region_summary = df.groupby('region').agg(
    total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
    classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
    high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum()),
    mean_aagr=('aagr_2024', lambda x: x[x != 'n.a.'].astype(float).mean())
).reset_index()

# Calculate share of high-growth firms (among classified)
region_summary['high_growth_share'] = region_summary['high_growth_firms'] / region_summary['classified_firms']

# Sort by number of firms descending
region_summary = region_summary.sort_values('total_firms', ascending=False)

region_summary.columns = ['region', 'total_firms', 'classified_firms', 'high_growth_firms', 'mean_aagr', 'high_growth_share']

print("Region Summary:")
print(region_summary.to_string())
print(f"\nTotal regions: {len(region_summary)}")

Region Summary:
             region  total_firms  classified_firms  high_growth_firms  mean_aagr  high_growth_share
9              Wien        10066              3108                371   0.067447           0.119369
4    Oberosterreich         7884              3896                404   0.046620           0.103696
3  Niederosterreich         7118              3047                296   0.035005           0.097145
6        Steiermark         5966              2455                255   0.038121           0.103870
7             Tirol         4630              1794                153   0.029310           0.085284
5          Salzburg         3935              1711                178   0.026416           0.104033
2           Karnten         2691              1108                100   0.052083           0.090253
8        Vorarlberg         2290               916                 68   0.033396           0.074236
1        Burgenland         1248               505                 51   0.038378    

## 6. Create Size-Band Summary Table

In [8]:
# Create size bands from emp_2021_num
df_size = df.copy()
df_size['size_band'] = pd.cut(df_size['emp_2021_num'], 
                              bins=[10, 19, 49, 249, float('inf')], 
                              labels=['11–19', '20–49', '50–249', '250+'],
                              right=True)

size_summary = df_size.groupby('size_band').agg(
    total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
    classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
    high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum()),
    mean_aagr=('aagr_2024', lambda x: x[x != 'n.a.'].astype(float).mean())
).reset_index()

# Calculate share of high-growth firms (among classified)
size_summary['high_growth_share'] = size_summary['high_growth_firms'] / size_summary['classified_firms']

size_summary.columns = ['size_band', 'total_firms', 'classified_firms', 'high_growth_firms', 'mean_aagr', 'high_growth_share']

# Sort by size band order
size_order = ['11–19', '20–49', '50–249', '250+']
size_summary['size_band'] = pd.Categorical(size_summary['size_band'], categories=size_order, ordered=True)
size_summary = size_summary.sort_values('size_band')

print("Size-Band Summary:")
print(size_summary.to_string())

Size-Band Summary:
  size_band  total_firms  classified_firms  high_growth_firms  mean_aagr  high_growth_share
0     11–19        10890              5876                671   0.002862           0.114193
1     20–49        10781              6389                643  -0.001975           0.100642
2    50–249         5532              3609                333  -0.004170           0.092269
3      250+         1426              1045                 65   0.002451           0.062201


/var/folders/4_/pht_r21d0yg70lpf8ry01d300000gn/T/ipykernel_40286/2749363800.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  size_summary = df_size.groupby('size_band').agg(


## 7. Extract KPI Values

In [9]:
# Extract KPI values
kpis = {}

# Overall counts
kpis['total_firms'] = len(df)
kpis['classified_firms'] = (df['ConsistentHighGrowthFirm_2024'] != 'n.a.').sum()
kpis['unclassified_firms'] = (df['ConsistentHighGrowthFirm_2024'] == 'n.a.').sum()

# High-growth counts
classified_df_temp = df[df['ConsistentHighGrowthFirm_2024'] != 'n.a.']
kpis['high_growth_firms'] = (classified_df_temp['ConsistentHighGrowthFirm_2024'] == 1).sum()
kpis['not_high_growth_firms'] = (classified_df_temp['ConsistentHighGrowthFirm_2024'] == 0).sum()

# Percentages
kpis['classified_pct'] = (kpis['classified_firms'] / kpis['total_firms']) * 100
kpis['high_growth_pct'] = (kpis['high_growth_firms'] / kpis['classified_firms']) * 100 if kpis['classified_firms'] > 0 else 0

# Mean AAGR
valid_aagr = pd.to_numeric(df['aagr_2024'], errors='coerce')
kpis['mean_aagr'] = valid_aagr.mean()
kpis['median_aagr'] = valid_aagr.median()

# Regional highlight (Vienna)
vienna_data = region_summary[region_summary['region'] == 'Wien']
if not vienna_data.empty:
    kpis['vienna_high_growth_share'] = vienna_data['high_growth_share'].values[0]
    kpis['vienna_high_growth_firms'] = int(vienna_data['high_growth_firms'].values[0])
    kpis['vienna_total_firms'] = int(vienna_data['total_firms'].values[0])

print("KPI Values:")
print("=" * 50)
for key, value in kpis.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2%}" if key.endswith('_pct') or key.endswith('_share') else f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value:,}" if isinstance(value, int) else f"{key}: {value}")

# Convert KPIs to DataFrame for easier saving
kpis_df = pd.DataFrame([kpis])
print("\nKPI DataFrame shape:", kpis_df.shape)

KPI Values:
total_firms: 46,085
classified_firms: 18629
unclassified_firms: 27456
high_growth_firms: 1883
not_high_growth_firms: 16746
classified_pct: 4042.31%
high_growth_pct: 1010.79%
mean_aagr: 0.0430
median_aagr: 0.0000
vienna_high_growth_share: 11.94%
vienna_high_growth_firms: 371
vienna_total_firms: 10,066

KPI DataFrame shape: (1, 12)


## 8. Save Processed Data

In [10]:
# Create processed data directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save master company-level table
master_table_path = '../data/processed/master_company_table.pkl'
master_table.to_pickle(master_table_path)
print(f"✓ Master table saved to {master_table_path}")

# Save as CSV as well for easier inspection
master_table_csv = '../data/processed/master_company_table.csv'
master_table.to_csv(master_table_csv, index=False)
print(f"✓ Master table (CSV) saved to {master_table_csv}")

# Save industry summary
industry_summary_path = '../data/processed/industry_summary.pkl'
industry_summary.to_pickle(industry_summary_path)
print(f"✓ Industry summary saved to {industry_summary_path}")

# Save industry summary as CSV
industry_summary_csv = '../data/processed/industry_summary.csv'
industry_summary.to_csv(industry_summary_csv, index=False)
print(f"✓ Industry summary (CSV) saved to {industry_summary_csv}")

# Save region summary
region_summary_path = '../data/processed/region_summary.pkl'
region_summary.to_pickle(region_summary_path)
print(f"✓ Region summary saved to {region_summary_path}")

# Save region summary as CSV
region_summary_csv = '../data/processed/region_summary.csv'
region_summary.to_csv(region_summary_csv, index=False)
print(f"✓ Region summary (CSV) saved to {region_summary_csv}")

# Save size summary
size_summary_path = '../data/processed/size_summary.pkl'
size_summary.to_pickle(size_summary_path)
print(f"✓ Size summary saved to {size_summary_path}")

# Save size summary as CSV
size_summary_csv = '../data/processed/size_summary.csv'
size_summary.to_csv(size_summary_csv, index=False)
print(f"✓ Size summary (CSV) saved to {size_summary_csv}")

# Save KPI values
kpis_path = '../data/processed/kpis.pkl'
kpis_df.to_pickle(kpis_path)
print(f"✓ KPI values saved to {kpis_path}")

# Save KPI values as CSV
kpis_csv = '../data/processed/kpis.csv'
kpis_df.to_csv(kpis_csv, index=False)
print(f"✓ KPI values (CSV) saved to {kpis_csv}")

# Save KPI dict as JSON for easy dashboard access
import json
kpis_json = '../data/processed/kpis.json'
with open(kpis_json, 'w') as f:
    # Convert numpy types to Python types for JSON serialization
    kpis_json_safe = {k: float(v) if isinstance(v, (np.floating, np.integer)) else v for k, v in kpis.items()}
    json.dump(kpis_json_safe, f, indent=2)
print(f"✓ KPI values (JSON) saved to {kpis_json}")

print("\n" + "=" * 60)
print("ALL DATA PREPARATION COMPLETE!")
print("=" * 60)
print("\nSaved files ready for dashboard:")
print("  • data/processed/master_company_table.pkl (+ .csv)")
print("  • data/processed/industry_summary.pkl (+ .csv)")
print("  • data/processed/region_summary.pkl (+ .csv)")
print("  • data/processed/size_summary.pkl (+ .csv)")
print("  • data/processed/kpis.pkl (+ .csv + .json)")
print("\nDashboard can now load these tables directly without reprocessing!")

✓ Master table saved to ../data/processed/master_company_table.pkl
✓ Master table (CSV) saved to ../data/processed/master_company_table.csv
✓ Industry summary saved to ../data/processed/industry_summary.pkl
✓ Industry summary (CSV) saved to ../data/processed/industry_summary.csv
✓ Region summary saved to ../data/processed/region_summary.pkl
✓ Region summary (CSV) saved to ../data/processed/region_summary.csv
✓ Size summary saved to ../data/processed/size_summary.pkl
✓ Size summary (CSV) saved to ../data/processed/size_summary.csv
✓ KPI values saved to ../data/processed/kpis.pkl
✓ KPI values (CSV) saved to ../data/processed/kpis.csv
✓ KPI values (JSON) saved to ../data/processed/kpis.json

ALL DATA PREPARATION COMPLETE!

Saved files ready for dashboard:
  • data/processed/master_company_table.pkl (+ .csv)
  • data/processed/industry_summary.pkl (+ .csv)
  • data/processed/region_summary.pkl (+ .csv)
  • data/processed/size_summary.pkl (+ .csv)
  • data/processed/kpis.pkl (+ .csv + .json